In [ ]:
import pandas as pd

In [ ]:
# load cross sectional results
estimation_results = pd.read_csv(r"C:\Users\jonat\Lasso_paper\Empirical\scripts\lasso_11_2025\cross_sectional_lasso_results.csv")

codes = pd.read_csv(r"C:\Users\jonat\Downloads\codes.csv")

In [ ]:
# rename column 1 to PERMNO
estimation_results.rename(columns={estimation_results.columns[0]: "PERMNO"}, inplace=True)    

In [ ]:
estimation_results_clean = estimation_results.copy()

# Keep PERMNO separately
permno = estimation_results_clean[["PERMNO"]]

# Select only Lasso columns
lasso = estimation_results_clean.filter(like="Lasso_")

# Clean column names
lasso.columns = (
    lasso.columns
        .str.replace("Lasso_", "", regex=False)
        .str.replace(r"_lag_.*", "", regex=True)
)

# Aggregate lags per topic
lasso_agg = lasso.T.groupby(level=0).sum().T

# Reattach PERMNO as first column
estimation_results = pd.concat([permno, lasso_agg], axis=1)


In [ ]:
estimation_results

In [ ]:
# merge estimation results on PERMNO with codes and drop rows in codes that have no match in estimation results
# make codes columns appear first
merged = pd.merge(codes, estimation_results, on="PERMNO", how="inner")

In [ ]:
# ------------------------------------------------------------
# SIC CLASSIFICATION BASED ON OFFICIAL DIVISIONS AND MAJOR GROUPS
# ------------------------------------------------------------
# Division A: Agriculture, Forestry, And Fishing (01–09)
# Division B: Mining (10–14)
# Division C: Construction (15–17)
# Division D: Manufacturing (20–39)
# Division E: Transportation, Communications, Electric, Gas, And Sanitary Services (40–49)
# Division F: Wholesale Trade (50–51)
# Division G: Retail Trade (52–59)
# Division H: Finance, Insurance, And Real Estate (60–67)
# Division I: Services (70–89)
# Division J: Public Administration (91–99)
# ------------------------------------------------------------

In [ ]:
# Ensure numeric
merged["HSICCD"] = pd.to_numeric(merged["HSICCD"], errors="coerce")

# Extract 2-digit major group
merged["sic2"] = (merged["HSICCD"] // 100).astype("Int64")


# --------------------------
# Division classification
# --------------------------

def classify_division(sic2):
    if pd.isna(sic2):
        return np.nan
    
    if 1 <= sic2 <= 9:
        return "A: Agriculture, Forestry, And Fishing"
    elif 10 <= sic2 <= 14:
        return "B: Mining"
    elif 15 <= sic2 <= 17:
        return "C: Construction"
    elif 20 <= sic2 <= 39:
        return "D: Manufacturing"
    elif 40 <= sic2 <= 49:
        return "E: Transportation, Communications, Utilities"
    elif 50 <= sic2 <= 51:
        return "F: Wholesale Trade"
    elif 52 <= sic2 <= 59:
        return "G: Retail Trade"
    elif 60 <= sic2 <= 67:
        return "H: Finance, Insurance, And Real Estate"
    elif 70 <= sic2 <= 89:
        return "I: Services"
    elif 91 <= sic2 <= 99:
        return "J: Public Administration"
    else:
        return "Unclassified"

merged["division"] = merged["sic2"].apply(classify_division)


# --------------------------
# Major group label (2-digit)
# --------------------------

major_group_dict = {
    # Division A
    1: "01: Agricultural Production Crops",
    2: "02: Agriculture Production Livestock",
    7: "07: Agricultural Services",
    8: "08: Forestry",
    9: "09: Fishing, Hunting, And Trapping",
    
    # Division B
    10: "10: Metal Mining",
    12: "12: Coal Mining",
    13: "13: Oil And Gas Extraction",
    14: "14: Nonmetallic Minerals Mining",
    
    # Division C
    15: "15: Building Construction",
    16: "16: Heavy Construction",
    17: "17: Special Trade Contractors",
    
    # Division D
    **{i: f"{i}: Manufacturing Major Group" for i in range(20, 40)},
    
    # Division E
    **{i: f"{i}: Transportation/Utilities Major Group" for i in range(40, 50)},
    
    # Division F
    50: "50: Wholesale Durable Goods",
    51: "51: Wholesale Nondurable Goods",
    
    # Division G
    **{i: f"{i}: Retail Trade Major Group" for i in range(52, 60)},
    
    # Division H
    60: "60: Depository Institutions",
    61: "61: Nondepository Credit Institutions",
    62: "62: Security And Commodity Brokers",
    63: "63: Insurance Carriers",
    64: "64: Insurance Agents And Brokers",
    65: "65: Real Estate",
    67: "67: Holding And Investment Offices",
    
    # Division I
    **{i: f"{i}: Services Major Group" for i in range(70, 90)},
    
    # Division J
    **{i: f"{i}: Public Administration Major Group" for i in range(91, 100)},
}

merged["major_group"] = merged["sic2"].map(major_group_dict)

In [55]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist


def euclidean_distance_stats(
    df: pd.DataFrame,
    topic_cols=None,
    standardize=True
):
    """
    Compute mean and std of pairwise Euclidean distances.
    
    Parameters
    ----------
    df : DataFrame
        Data containing topic counts.
    topic_cols : list or None
        Explicit topic columns. If None, uses all numeric columns
        except common ID variables.
    standardize : bool
        If True, z-score columns before computing distance.
    """

    # -----------------------
    # 1) Select topic columns
    # -----------------------
    if topic_cols is None:
        X = df.select_dtypes(include=[np.number]).copy()
        X = X.drop(columns=["PERMNO", "HSICCD", "sic2"], errors="ignore")
    else:
        X = df[topic_cols].copy()

    # Force numeric
    X = X.apply(pd.to_numeric, errors="coerce")

    # Drop rows with missing values
    X = X.dropna(axis=0, how="any")

    # Drop constant columns
    col_std = X.std(axis=0, ddof=0)
    X = X.loc[:, col_std > 0]

    if X.shape[0] < 2:
        return {"n_rows": X.shape[0], "n_cols": X.shape[1], "mean": np.nan, "std": np.nan}

    # -----------------------
    # 2) Optional standardize
    # -----------------------
    if standardize:
        X = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0)

    X = X.astype(np.float64)

    # -----------------------
    # 3) Euclidean distances
    # -----------------------
    d = pdist(X.values, metric="euclidean")

    return {
        "n_rows": X.shape[0],
        "n_cols": X.shape[1],
        "mean": float(d.mean()),
        "std": float(d.std())
    }


# -------------------------------------------------
# FULL SAMPLE
# -------------------------------------------------
full_stats = euclidean_distance_stats(merged, standardize=True)

print("FULL SAMPLE")
print("n_rows:", full_stats["n_rows"])
print("n_cols:", full_stats["n_cols"])
print("Mean pairwise Euclidean distance:", full_stats["mean"])
print("Std pairwise Euclidean distance:", full_stats["std"])
print()


# -------------------------------------------------
# BY DIVISION
# -------------------------------------------------
results = []

for div, g in merged.groupby("division", dropna=False):
    stats = euclidean_distance_stats(g, standardize=True)
    results.append({
        "division": div,
        "n_rows": stats["n_rows"],
        "n_cols": stats["n_cols"],
        "euclid_mean": stats["mean"],
        "euclid_std": stats["std"]
    })

by_division = pd.DataFrame(results).sort_values("division").reset_index(drop=True)

print("BY DIVISION")
print(by_division)



# ---------------------------------------
# DIFFERENCE RELATIVE TO FULL SAMPLE
# ---------------------------------------

overall_mean = full_stats["mean"]
overall_std  = full_stats["std"]

diff_table = by_division.copy()

# Absolute differences
diff_table["diff_mean_vs_full"] = diff_table["euclid_mean"] - overall_mean
diff_table["diff_std_vs_full"]  = diff_table["euclid_std"]  - overall_std

# Percentage differences
diff_table["pct_diff_mean"] = (
    diff_table["diff_mean_vs_full"] / overall_mean
) * 100

diff_table["pct_diff_std"] = (
    diff_table["diff_std_vs_full"] / overall_std
) * 100

# Optional: round for readability
diff_table = diff_table.round(4)

print("DIFFERENCE VS FULL SAMPLE")
print(diff_table[
    [
        "division",
        "euclid_mean",
        "diff_mean_vs_full"
    ]
])



FULL SAMPLE
n_rows: 1620
n_cols: 182
Mean pairwise Euclidean distance: 15.453220722938685
Std pairwise Euclidean distance: 11.19923209109452

BY DIVISION
                                       division  n_rows  n_cols  euclid_mean  \
0         A: Agriculture, Forestry, And Fishing       2     181    26.907248   
1                                     B: Mining     111     182    17.788695   
2                               C: Construction      26     182    18.277179   
3                              D: Manufacturing     419     182    16.315786   
4  E: Transportation, Communications, Utilities     170     182    15.307707   
5                            F: Wholesale Trade      46     182    15.978478   
6                               G: Retail Trade      70     182    16.852718   
7        H: Finance, Insurance, And Real Estate     638     182    13.002116   
8                                   I: Services     137     182    16.732614   
9                      J: Public Administratio

In [54]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist

def euclidean_distance_stats(df: pd.DataFrame, topic_cols=None, standardize=True):
    """
    Mean/std of pairwise Euclidean distances on topic-count columns.
    Drops ID vars if topic_cols is None.
    """
    if topic_cols is None:
        X = df.select_dtypes(include=[np.number]).copy()
        X = X.drop(columns=["PERMNO", "HSICCD", "sic2"], errors="ignore")
    else:
        X = df.loc[:, topic_cols].copy()

    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.dropna(axis=0, how="any")

    # drop constant topic columns
    col_std = X.std(axis=0, ddof=0)
    X = X.loc[:, col_std > 0]

    if X.shape[0] < 2 or X.shape[1] == 0:
        return {"n_rows": int(X.shape[0]), "n_cols": int(X.shape[1]), "mean": np.nan, "std": np.nan}

    if standardize:
        X = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0)

    d = pdist(X.to_numpy(dtype=np.float64), metric="euclidean")
    return {"n_rows": int(X.shape[0]), "n_cols": int(X.shape[1]), "mean": float(d.mean()), "std": float(d.std())}


# ---------------------------------------
# FULL SAMPLE (same as before)
# ---------------------------------------
full_stats = euclidean_distance_stats(merged, standardize=True)

print("FULL SAMPLE")
print("n_rows:", full_stats["n_rows"])
print("n_cols:", full_stats["n_cols"])
print("Mean pairwise Euclidean distance:", full_stats["mean"])
print("Std pairwise Euclidean distance:", full_stats["std"])
print()


# ---------------------------------------
# BY MAJOR GROUP
# ---------------------------------------
rows = []
for mg, g in merged.groupby("major_group", dropna=False):
    stats = euclidean_distance_stats(g, standardize=True)
    rows.append({
        "major_group": mg,
        "n_rows": stats["n_rows"],
        "n_cols": stats["n_cols"],
        "euclid_mean": stats["mean"],
        "euclid_std": stats["std"],
    })

by_major_group = (
    pd.DataFrame(rows)
      .sort_values(["n_rows", "major_group"], ascending=[False, True])
      .reset_index(drop=True)
)

print("BY MAJOR GROUP")
print(by_major_group)
print()


# ---------------------------------------
# DIFFERENCE VS FULL SAMPLE
# ---------------------------------------
overall_mean = full_stats["mean"]
overall_std  = full_stats["std"]

diff_major_group = by_major_group.copy()
diff_major_group["diff_mean_vs_full"] = diff_major_group["euclid_mean"] - overall_mean
diff_major_group["pct_diff_mean"] = (diff_major_group["diff_mean_vs_full"] / overall_mean) * 100

diff_major_group["diff_std_vs_full"] = diff_major_group["euclid_std"] - overall_std
diff_major_group["pct_diff_std"] = (diff_major_group["diff_std_vs_full"] / overall_std) * 100

diff_major_group = diff_major_group.round(4)

diff_major_group = diff_major_group[diff_major_group["n_rows"] >= 10]

print("DIFFERENCE VS FULL SAMPLE (MAJOR GROUP)")
print(diff_major_group[
    [
        "major_group",
        "n_rows",
        "euclid_mean",
        "diff_mean_vs_full"
    ]
])


FULL SAMPLE
n_rows: 1620
n_cols: 182
Mean pairwise Euclidean distance: 15.453220722938685
Std pairwise Euclidean distance: 11.19923209109452

BY MAJOR GROUP
                                 major_group  n_rows  n_cols  euclid_mean  \
0         67: Holding And Investment Offices     443     182    10.970194   
1                 13: Oil And Gas Extraction      83     182    17.608998   
2   49: Transportation/Utilities Major Group      83     182    12.985207   
3              28: Manufacturing Major Group      66     182    16.207932   
4                   73: Services Major Group      61     182    16.808186   
..                                       ...     ...     ...          ...   
61         01: Agricultural Production Crops       1       0          NaN   
62                 07: Agricultural Services       1       0          NaN   
63                  75: Services Major Group       1       0          NaN   
64                  86: Services Major Group       1       0          NaN